<a href="https://colab.research.google.com/github/GSANCHEZH-UC/Clase3.Multiprovesamiento/blob/main/20260406_CLASE_3_1_MINGDATOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PROGRAMACIÓN EN PAYTHON PARA INGENIERÍA DE DATOS**

# **2026.04.06. Clase 3.  Aplicando procesamiento parelelo**

In [ ]:
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime


# **Preparación del acceso al Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Tomando la ruta actual del GDrive del usuario
current_dir = os.getcwd()

# Cosntruyecdo el path para la librería utilidadesML, se asume que está en la raía del GDrive del usuario
Mi_path = os.path.join(current_dir, "drive/MyDrive/Colab Notebooks")
sys.path.append(Mi_path)

import utilidadesML as ml

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Carga de datos usando paralelisto**

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor # clase para hacer procesamiento paralelo
from google.colab import drive

drive.mount('/content/drive')

# Función a ajecutar en procesos paralelos
def cargar_archivo(info):
    nombre, ruta = info
    try:
        if os.path.exists(ruta):
            df = pd.read_csv(ruta, sep=",") # en df queda el DataSer leído del archivo "nombre"
            print(f"Cargado correctamente: {nombre}")
            return nombre, df
        else:
            print(f"Archivo no encontrado: {ruta}")
            return nombre, None
    except Exception as e:
        print(f"Error al cargar {nombre}: {e}")
        return nombre, None

# Definir ruta de acceso a los archicos de datos
base_path = '/content/drive/MyDrive/Colab Notebooks/Chocolate Sales Dataset 2023 - 2024/'

# Lista de archivos a cargar
archivos_a_cargar = [
    ('ventas', os.path.join(base_path, 'sales.csv')),
    ('clientes', os.path.join(base_path, 'customers.csv')),
    ('productos', os.path.join(base_path, 'products.csv')),
    ('tiendas', os.path.join(base_path, 'stores.csv')),
    ('calendario', os.path.join(base_path, 'calendar.csv'))
]

# Uso de ThreadPoolExecutor para hacer varios proceso de carga a la vez
resultados = {}
with ThreadPoolExecutor(max_workers=5) as executor: # max_workers define el número de procesos en paralelo
    # executor.map lanza todos los proceso con la llamada a cargar_archivo
    # cada proceso toma una de las tuplas de archivos_a_cargar
    # en  lista_resultados queda la lista de los DataSets devueltos por la función cargar_archivo
    lista_resultados = list(executor.map(cargar_archivo, archivos_a_cargar))

# Se asigna cada DataSet a una variable global con el nombre correspondiente
for nombre, df in lista_resultados:
    if df is not None:
        globals()[nombre] = df
        resultados[nombre] = df

print("\nProceso de carga finalizado.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargado correctamente: tiendas
Cargado correctamente: productos
Cargado correctamente: calendario
Cargado correctamente: clientes
Cargado correctamente: ventas

Proceso de carga finalizado.


# **Mostrando las características de los datos**

**Explorando los datos**

In [ ]:
print(f"Ventas: {ventas.shape}\n")
print(f"Clientes: {clientes.shape}\n")
print(f"Productos: {productos.shape}\n")
print(f"Tiendas: {tiendas.shape}\n")
print(f"Calendario: {calendario.shape}\n")



Ventas: (1000000, 11)

Clientes: (50000, 5)

Productos: (200, 6)

Tiendas: (100, 5)

Calendario: (731, 6)



In [ ]:
print(f"Ventas: {ventas.columns}\n")
print(f"Clientes: {clientes.columns}\n")
print(f"Productos: {productos.columns}\n")
print(f"Tiendas: {tiendas.columns}\n")
print(f"Calendario: {calendario.columns}\n")

Ventas: Index(['order_id', 'order_date', 'product_id', 'store_id', 'customer_id',
       'quantity', 'unit_price', 'discount', 'revenue', 'cost', 'profit'],
      dtype='object')

Clientes: Index(['customer_id', 'age', 'gender', 'loyalty_member', 'join_date'], dtype='object')

Productos: Index(['product_id', 'product_name', 'brand', 'category', 'cocoa_percent',
       'weight_g'],
      dtype='object')

Tiendas: Index(['store_id', 'store_name', 'city', 'country', 'store_type'], dtype='object')

Calendario: Index(['date', 'year', 'month', 'day', 'week', 'day_of_week'], dtype='object')



In [ ]:
print(f"Ventas:\n {ventas.dtypes}\n")
print(f"Clientes:\n {clientes.dtypes}\n")
print(f"Productos:\n {productos.dtypes}\n")
print(f"Tiendas:\n {tiendas.dtypes}\n")
print(f"Calendario:\n {calendario.dtypes}\n")

Ventas:
 order_id        object
order_date      object
product_id      object
store_id        object
customer_id     object
quantity         int64
unit_price     float64
discount       float64
revenue        float64
cost           float64
profit         float64
dtype: object

Clientes:
 customer_id       object
age                int64
gender            object
loyalty_member     int64
join_date         object
dtype: object

Productos:
 product_id       object
product_name     object
brand            object
category         object
cocoa_percent     int64
weight_g          int64
dtype: object

Tiendas:
 store_id      object
store_name    object
city          object
country       object
store_type    object
dtype: object

Calendario:
 date           object
year            int64
month           int64
day             int64
week            int64
day_of_week     int64
dtype: object



In [ ]:
print(f"Ventas:\n {ventas.info()}\n")
print(f"Clientes:\n {clientes.info()}\n")
print(f"Productos:\n {productos.info()}\n")
print(f"Tiendas:\n {tiendas.info()}\n")
print(f"Calendario:\n {calendario.info()}\n")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 11 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   order_id     1000000 non-null  object 
 1   order_date   1000000 non-null  object 
 2   product_id   1000000 non-null  object 
 3   store_id     1000000 non-null  object 
 4   customer_id  1000000 non-null  object 
 5   quantity     1000000 non-null  int64  
 6   unit_price   1000000 non-null  float64
 7   discount     1000000 non-null  float64
 8   revenue      1000000 non-null  float64
 9   cost         1000000 non-null  float64
 10  profit       1000000 non-null  float64
dtypes: float64(5), int64(1), object(5)
memory usage: 83.9+ MB
Ventas:
 None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customer_id     50000 non-null  objec

In [ ]:
ventas['order_date'] = pd.to_datetime(ventas['order_date'])
clientes['join_date'] = pd.to_datetime(clientes['join_date'])
calendario['date'] = pd.to_datetime(calendario['date'])

# Unir ventas con productos
df_completo = pd.merge(ventas, productos, on='product_id', how='left')

# Unir el resultado con clientes
df_completo = pd.merge(df_completo, clientes, on='customer_id', how='left')

# Unir el resultado con tiendas
df_completo = pd.merge(df_completo, tiendas, on='store_id', how='left')

# Unir el resultado con calendario por la fecha
df_completo = pd.merge(df_completo, calendario, left_on='order_date', right_on='date', how='left')

df_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 30 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   order_id        1000000 non-null  object        
 1   order_date      1000000 non-null  datetime64[ns]
 2   product_id      1000000 non-null  object        
 3   store_id        1000000 non-null  object        
 4   customer_id     1000000 non-null  object        
 5   quantity        1000000 non-null  int64         
 6   unit_price      1000000 non-null  float64       
 7   discount        1000000 non-null  float64       
 8   revenue         1000000 non-null  float64       
 9   cost            1000000 non-null  float64       
 10  profit          1000000 non-null  float64       
 11  product_name    990236 non-null   object        
 12  brand           990236 non-null   object        
 13  category        990236 non-null   object        
 14  cocoa_percent   990

In [ ]:
print("Valores nulos por columna numérica:")
print(df_completo.select_dtypes(include='number').isnull().sum())

Valores nulos por columna numérica:
quantity             0
unit_price           0
discount             0
revenue              0
cost                 0
profit               0
cocoa_percent     9764
weight_g          9764
age                  0
loyalty_member       0
year                 0
month                0
day                  0
week                 0
day_of_week          0
dtype: int64


In [ ]:
mediacocoa_percent = df_completo['cocoa_percent'].mean()
print(f"La media de la columna 'cocoa_percent' es: {mediacocoa_percent}")
df_completo['cocoa_percent'] = df_completo['cocoa_percent'].fillna(mediacocoa_percent)
print(f"Valores nulos en 'cocoa_percent' después de rellenar con la moda: {df_completo['cocoa_percent'].isnull().sum()}")

La media de la columna 'cocoa_percent' es: 69.14813236440607
Valores nulos en 'cocoa_percent' después de rellenar con la moda: 0


In [ ]:
mediaweight_g = df_completo['weight_g'].mean()
print(f"La media de la columna 'weight_g' es: {mediaweight_g}")
df_completo['weight_g'] = df_completo['weight_g'].fillna(mediaweight_g)
print(f"Valores nulos en 'weight_g' después de rellenar con la moda: {df_completo['cocoa_percent'].isnull().sum()}")

La media de la columna 'weight_g' es: 107.4314001914695
Valores nulos en 'weight_g' después de rellenar con la moda: 0


In [ ]:
print("Valores nulos por columna numérica:")
print(df_completo.select_dtypes(include='number').isnull().sum())

Valores nulos por columna numérica:
quantity          0
unit_price        0
discount          0
revenue           0
cost              0
profit            0
cocoa_percent     0
weight_g          0
age               0
loyalty_member    0
year              0
month             0
day               0
week              0
day_of_week       0
dtype: int64
